## Sequencial chain

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser

prompt1 = PromptTemplate(
    template = "write a eassy on this topic  {topic} under 500 words"
)

prompt2  = PromptTemplate(
    template = "{instructions} check and return only no. out of 10 to this eassy {essay}"
)

model = ChatGroq( model = "openai/gpt-oss-20b")

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

class Score(BaseModel):
    score: int
    essay : str

pydanticParser = PydanticOutputParser(pydantic_object=Score)

chain =  prompt1 | model |parser | \
RunnableLambda(lambda essay: {"essay": essay, "instructions":pydanticParser.get_format_instructions()}) \
| prompt2 | model | pydanticParser



In [13]:
res = chain.invoke({"topic":"cow"})

In [15]:
print(res.essay)

The Cow: A Gentle Giant of Human History

The cow (Bos taurus) is one of the most iconic domesticated animals in human history. From the plains of the Indus Valley to the pastures of the American Midwest, cattle have shaped societies, economies, and cultures for millennia. Though often seen simply as a source of milk and meat, the cow’s influence runs far deeper, touching agriculture, religion, art, and even the environment.

**Biology and Domestication**

Cattle are large herbivorous mammals belonging to the Bovidae family. They are ruminants, meaning they digest cellulose through a specialized four‑compartment stomach. This adaptation allows them to thrive on grasses and other low‑quality forage that many other animals cannot utilize efficiently. The domestication of the cow is believed to have begun around 10,000 BC in the Fertile Crescent, when humans first began to herd wild aurochs. Over time, selective breeding produced the many breeds we see today—Holsteins for milk, Angus for 

## Parallel Chain

In [ ]:
from langchain_core.runnables import RunnableParallel 

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

prompt = PromptTemplate(
    template = "give me one fun facts about  {topic}"
    )
prompt2  = PromptTemplate(
    template = "{instructions} summarize the following facts into one sentence: {first} \n {second}"
)

model = ChatGroq( model = "openai/gpt-oss-20b")
parser = StrOutputParser()

parallel_chain = RunnableParallel(
    {"first": prompt | model | parser,
     "second": prompt | model | parser}
)

class Summary(BaseModel):
    summary: str
    first: str
    second: str

pydanticParser = PydanticOutputParser(pydantic_object=Summary)
# merged = (
#     parallel_chain
#     | RunnableLambda(lambda x: {
#         **x,
#         "summary": (prompt2 | model | pydanticParser).invoke({"first": x["first"],
#                          "second": x["second"],"instructions": pydanticParser.get_format_instructions()})
#         })
#     )

merged = (
    parallel_chain|RunnableLambda(lambda x: {
        "first": x["first"],
        "second": x["second"],
        "instructions": pydanticParser.get_format_instructions(),
    }) \
    |prompt2 | model | pydanticParser
    )




In [32]:
res = merged.invoke({"topic":"elephant"})

In [33]:
res

Summary(summary='An elephant’s trunk is a super‑flexible 2‑meter nose‑arm that can pick up a grain of rice, lift a small car, or caress a newborn, while its 22‑month pregnancy—the longest among land animals—allows the calf to develop a massive brain and complex social skills.', first='An elephant’s trunk is basically a super‑flexible, 2‑meter‑long nose‑arm that combines a nose and a tongue into one! It can pick up a single grain of rice, lift a small car, or gently caress a newborn baby—making it one of the most versatile “tools” in the animal kingdom.', second='An adult elephant’s pregnancy lasts about 22 months— the longest gestation period of any land animal! This extended time allows the calf to develop its massive brain and complex social skills before it’s born.')

## Conditional Chain


In [19]:
from langchain_core.runnables import RunnableBranch,RunnableLambda
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

class SentimentResponse(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The sentiment of the text")
    
prompt = PromptTemplate(
    template = "{instruction} give the sentiment of the following text: {text} "
)
model = ChatGroq( model = "openai/gpt-oss-20b")
parser = PydanticOutputParser(pydantic_object=SentimentResponse)



chain1 = prompt | model | parser | RunnableLambda(lambda x: {"sentiment": x.sentiment})
conditional_chain = RunnableBranch(
    (lambda x: x["sentiment"] == "positive",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. Great job!")),
    (lambda x: x["sentiment"] == "negative",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. Try to be more positive!")),
    (lambda x: x["sentiment"] == "neutral",
        RunnableLambda(lambda x: f"The text is {x['sentiment']}. It's balanced.")),
    RunnableLambda(lambda x: "Sentiment could not be determined.")
)

chain = chain1 | conditional_chain


In [20]:
res = chain.invoke({"instruction":parser.get_format_instructions(), "text":"indias capital is delhi!"})
res

"The text is neutral. It's balanced."

In [9]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Lambda |        
       +--------+        
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +-------